In [ ]:
import numpy as np
import torch
import torch.nn as nn
import tqdm
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score



In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import pandas as pd

base_path = "/content/drive/MyDrive/dev_phase/subtask1/train/"
files = glob.glob(os.path.join(base_path, "*.csv"))

data = {}

for file in files:
    lang = os.path.splitext(os.path.basename(file))[0]  # amh, arb, eng

    df = pd.read_csv(file)

    data[lang] = {
        "X": df["text"].tolist(),
        "y": df["polarization"].tolist(),
        "df": df
    }

print("Loaded languages:", sorted(data.keys()))


In [ ]:
model_name = "BAAI/bge-m3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
embedding_model = AutoModel.from_pretrained(model_name).to(device)
embedding_model.eval()

for p in embedding_model.parameters():
    p.requires_grad = False

def mean_pooling(model_output, attention_mask):
    token_embeds = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    sum_embeddings = torch.sum(token_embeds * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

In [ ]:
def get_all_embeddings(texts, model, tokenizer, device, batch_size=32):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**enc)
            embeddings = mean_pooling(outputs, enc['attention_mask'])
        all_embs.append(embeddings.cpu())
    return torch.cat(all_embs, dim=0)


embeddings_by_lang = {}

for lang, content in data.items():   # data[lang] has "X" and "y"
    print(f"Embedding language: {lang}")

    X_text = content["X"]
    y_labels = content["y"]

    X_emb = get_all_embeddings(X_text, embedding_model, tokenizer, device)
    y_tensor = torch.tensor(y_labels, dtype=torch.long)

    embeddings_by_lang[lang] = {
        "X": X_emb,
        "y": y_tensor
    }



Embedding: 100%|██████████| 134/134 [00:31<00:00,  4.30it/s]


In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_embeddings, y_tensor, test_size=0.3, random_state=42, stratify=y_tensor
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, embed_dim, num_classes):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear((embed_dim // 4) * 256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x: [B, embed_dim]
        x = x.unsqueeze(1)        # [B, 1, embed_dim]
        x = self.conv(x)          # [B, 256, embed_dim/4]
        x = x.flatten(1)          # [B, *]
        return self.fc(x)


In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


Loaded classifier!


In [ ]:
def evaluate_language_cv_cnn(
    X_embeddings,
    y_tensor,
    device,
    num_classes,
    k=5,
    epochs=5,
    batch_size=32
):

    X = X_embeddings.numpy()
    y = y_tensor.numpy()

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    all_acc = []
    all_macro_f1 = []


Epoch 1/20 - Loss: 3.1165 - Val Macro F1: 0.8519
Epoch 2/20 - Loss: 2.6700 - Val Macro F1: 0.8520
Epoch 3/20 - Loss: 2.5103 - Val Macro F1: 0.8551
Epoch 4/20 - Loss: 2.2634 - Val Macro F1: 0.8551
Epoch 5/20 - Loss: 2.0580 - Val Macro F1: 0.8504
Epoch 6/20 - Loss: 1.9206 - Val Macro F1: 0.8566
Epoch 7/20 - Loss: 1.8139 - Val Macro F1: 0.8504
Epoch 8/20 - Loss: 1.5714 - Val Macro F1: 0.8535
Epoch 9/20 - Loss: 1.5428 - Val Macro F1: 0.8503
Epoch 10/20 - Loss: 1.3815 - Val Macro F1: 0.8535
Epoch 11/20 - Loss: 1.4080 - Val Macro F1: 0.8520
Epoch 12/20 - Loss: 1.3315 - Val Macro F1: 0.8504
Epoch 13/20 - Loss: 1.0633 - Val Macro F1: 0.8566
Epoch 14/20 - Loss: 1.0705 - Val Macro F1: 0.8534
Epoch 15/20 - Loss: 1.0559 - Val Macro F1: 0.8519
Epoch 16/20 - Loss: 0.8974 - Val Macro F1: 0.8472
Epoch 17/20 - Loss: 0.8795 - Val Macro F1: 0.8519
Epoch 18/20 - Loss: 0.8859 - Val Macro F1: 0.8535
Epoch 19/20 - Loss: 0.7348 - Val Macro F1: 0.8535
Epoch 20/20 - Loss: 0.6682 - Val Macro F1: 0.8566


In [ ]:
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
        X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_test_t  = torch.tensor(y_test, dtype=torch.long).to(device)

        train_loader = DataLoader(
            TensorDataset(X_train_t, y_train_t),
            batch_size=batch_size,
            shuffle=True
        )

        classifier = CNNClassifier(
            X_train_t.shape[1],
            num_classes
        ).to(device)

        total, trainable = count_params(classifier)
        print(f"[Fold {fold+1}] Total params: {total:,}")

        optimizer = torch.optim.AdamW(
            classifier.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )

        criterion = nn.CrossEntropyLoss()



=== Test Classification Report ===
              precision    recall  f1-score   support

           0     0.8530    0.8241    0.8383       324
           1     0.8267    0.8553    0.8408       318

    accuracy                         0.8396       642
   macro avg     0.8399    0.8397    0.8396       642
weighted avg     0.8400    0.8396    0.8395       642



In [ ]:
classifier.train()
for epoch in range(epochs):
    for bx, by in train_loader:
        optimizer.zero_grad()
        gits = classifier(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()


Saved fine-tuned classifier to classifier_zho_finetuned.pth


In [ ]:
classifier.eval()
with torch.no_grad():
    logits = classifier(X_test_t)
    y_pred = torch.argmax(logits, dim=1).cpu().numpy()
    acc = np.mean(y_pred == y_test)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    all_acc.append(acc)
    all_macro_f1.append(macro_f1)
